# Resolución de Prueba Técnica: Reconstrucción Tabular desde Formato HOCR
**Autor:** Jonathan Eduardo Castilla Zamora

## 1. Resumen
El presente *notebook* implementa una solución automatizada para la extracción y reconstrucción de estructuras tabulares a partir de archivos HOCR. El desafío principal radica en que el formato HOCR proporciona datos desestructurados espacialmente (texto puro y coordenadas de *bounding boxes*), careciendo de metadatos semánticos sobre filas y columnas.

## 2. Justificación Arquitectónica y Selección de Algoritmos
Para resolver este problema, se descartó el uso de heurísticas basadas en umbrales de píxeles fijos (*hardcoding*), ya que son frágiles ante variaciones en la resolución de escaneo o inclinación del documento. En su lugar, se propone un *pipeline* híbrido de dos etapas:



1. **Agrupamiento Espacial (Machine Learning No Supervisado):** Se seleccionó el algoritmo **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) sobre alternativas como K-Means.
   * *¿Por qué no K-Means?* K-Means requiere definir a priori el número de clústeres ($k$), lo cual es imposible al desconocer cuántas filas o columnas tiene una tabla arbitraria.
   * *¿Por qué DBSCAN?*  DBSCAN agrupa puntos basándose en la densidad local. Permite inferir dinámicamente el número de filas y columnas evaluando la proximidad de los centroides de las palabras, absorbiendo ligeras desviaciones (ruido) causadas por escaneos imperfectos.

2. **Refinamiento Semántico (Inteligencia Artificial Generativa):**
   Los motores OCR clásicos (como Tesseract) presentan una tasa de error de caracteres (CER) inherente, confundiendo frecuentemente caracteres visualmente similares (ej. '0' vs 'O', '1' vs 'l'). Se integró una capa opcional utilizando un Modelo de Lenguaje Grande (LLM) a través de la API de Gemini. El LLM no actúa como un simple corrector ortográfico, sino como un motor de inferencia probabilística que utiliza el contexto de la tabla reconstruida espacialmente para aplicar correcciones semánticas y estandarizar la salida a un formato CSV estricto.


In [25]:
# ==============================================================================
# CONFIGURACIÓN DEL ENTORNO Y DEPENDENCIAS
# ==============================================================================
# Instalación de los paquetes requeridos
!pip install -q beautifulsoup4 pandas scikit-learn google-generativeai

# Importaciones de la biblioteca estándar
import re
import os
import glob
import subprocess
import getpass
import platform

# Bibliotecas de terceros para procesamiento de datos y Machine Learning
import pandas as pd
import bs4
from bs4 import BeautifulSoup
import sklearn
from sklearn.cluster import DBSCAN

# Integración de IA Generativa
import google.generativeai as genai

# Módulos específicos de Colab para la gestión del entorno
from google.colab import drive, files
from IPython.display import display

print("--- REPORTE DE VERSIONES DEL ENTORNO ---")
print(f"Python:               {platform.python_version()}")
print(f"Pandas:               {pd.__version__}")
print(f"BeautifulSoup4:       {bs4.__version__}") # Corrected to use bs4.__version__
print(f"Scikit-learn (DBSCAN):{sklearn.__version__}")
print(f"Google Generative AI: {genai.__version__}")
print("----------------------------------------")

--- REPORTE DE VERSIONES DEL ENTORNO ---
Python:               3.12.12
Pandas:               2.2.2
BeautifulSoup4:       4.13.5
Scikit-learn (DBSCAN):1.6.1
Google Generative AI: 0.8.6
----------------------------------------


## 3. Configuración e Inicialización del Entorno (Plug & Play)
Para garantizar una evaluación técnica fluida y eliminar la fricción de configuraciones locales, este módulo prescinde del montaje de Google Drive. La función `setup_environment` inicializa un entorno efímero clonando directamente el repositorio desde GitHub. Esto asegura que los directorios de trabajo y los archivos `.hocr` de entrada se descarguen y configuren automáticamente antes de arrancar el *pipeline* principal, permitiendo una ejecución limpia de un solo clic.



In [32]:
# ==============================================================================
# --- 1. CONFIGURACIÓN E INICIALIZACIÓN
# ==============================================================================

# 1. Configuración del Repositorio Público
GITHUB_REPO_URL = "https://github.com/JonathanCastilla/hocr-tabular-reconstruction"
REPO_NAME = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

# 2. Configuración de rutas locales en la máquina virtual de Colab ()
BASE_DIR = f'/content/{REPO_NAME}'
PROJECT_FOLDER = os.path.join(BASE_DIR, 'files')
OUTPUT_DIRECTORY = '/content/Results'

def setup_environment() -> list:
    """
    Prepara el directorio de trabajo descargando los archivos fuente directamente
    desde GitHub, eliminando la necesidad de montar el Google Drive del usuario.

    Retorna:
        list: Rutas absolutas a los archivos .hocr descargados.
    """
    print("--- Inicializando Entorno de Trabajo ---")

    # Clonar el repositorio si no existe en la máquina virtual
    if not os.path.exists(BASE_DIR):
        print(f"Descargando archivos fuente desde GitHub...")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, BASE_DIR])
    else:
        print("Los archivos fuente ya están cargados en el entorno local.")

    # Creación del directorio temporal de resultados
    if not os.path.exists(OUTPUT_DIRECTORY):
        os.makedirs(OUTPUT_DIRECTORY)

    # Auditoría de archivos de entrada (.hocr)
    hocr_files_pattern = os.path.join(PROJECT_FOLDER, "*.hocr")
    hocr_file_list = glob.glob(hocr_files_pattern)

    if not hocr_file_list:
        print(f"⚠️ Advertencia: No se encontraron archivos .hocr en: {PROJECT_FOLDER}")
        print("Asegúrese de que la carpeta 'files' en su repositorio contenga los documentos.")
    else:
        print(f"✅ Se encontraron {len(hocr_file_list)} archivos .hocr listos para su procesamiento.")

    return hocr_file_list

# Ejecución de la inicialización
target_hocr_files = setup_environment()

--- Inicializando Entorno de Trabajo ---
Los archivos fuente ya están cargados en el entorno local.
✅ Se encontraron 3 archivos .hocr listos para su procesamiento.


## 4. Autorización de LLM y Selección de Modelo
Esta sección establece la conexión con la API de Google Generative AI. Consulta dinámicamente el catálogo de modelos disponibles para permitir flexibilidad. Se utiliza el módulo `getpass` para ingresar la API Key de forma segura sin exponer credenciales en el historial de ejecución del *notebook*.

In [27]:
# ==============================================================================
# AUTORIZACIÓN DE API Y CATÁLOGO DINÁMICO DE MODELOS
# ==============================================================================
print("Por favor, ingrese su Google Gemini API Key (la entrada estará oculta):")
api_key = getpass.getpass('API Key: ')

# Inicializar la configuración de la API
genai.configure(api_key=api_key)

print("\n--- MODELOS DE GENERACIÓN DE TEXTO DISPONIBLES ---")
available_models = []
try:
    # Obtener y filtrar los modelos que soportan generación de texto
    for model_info in genai.list_models():
        if 'generateContent' in model_info.supported_generation_methods:
            available_models.append(model_info.name)
            print(f"- {model_info.name}")
except Exception as api_error:
    print(f"Error al obtener el catálogo de modelos: {api_error}")

# Definir el modelo objetivo para la tarea de refinamiento semántico
# Se prefieren los modelos 'flash' para tareas de datos estructurados debido a su menor latencia
TARGET_LLM_MODEL = 'models/gemini-2.5-flash'
print(f"\nModelo seleccionado para el pipeline: {TARGET_LLM_MODEL}")

Por favor, ingrese su Google Gemini API Key (la entrada estará oculta):
API Key: ··········

--- MODELOS DE GENERACIÓN DE TEXTO DISPONIBLES ---
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-2.5-flash-lite-preview-09-2025
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-

## 5. Núcleo Algorítmico del Pipeline
El procesamiento central del documento se ha desacoplado en tres módulos independientes. Esta arquitectura modular (*Separation of Concerns*) facilita el mantenimiento, la depuración y la escalabilidad del código, permitiendo aislar fallos geométricos de fallos semánticos.

### 5.1. Módulo de Extracción Espacial (`extract_spatial_tokens`)
El formato HOCR anida la información posicional dentro de los atributos de las etiquetas HTML.

**Justificación Técnica:**
* **Análisis de Árbol DOM:** Se utiliza `BeautifulSoup` en lugar de expresiones regulares puras para leer el documento completo, ya que los motores OCR a menudo generan HTML mal formado que rompería un *parser* estricto.
* **Cálculo de Centroides:** En lugar de utilizar la coordenada superior (`y0`) o inferior (`y1`) de la caja delimitadora, se calcula el centroide vertical (`y_centroid`). Esto es una decisión de ingeniería crucial: mitiga la varianza geométrica introducida por palabras con diferentes alturas de fuente (ej. letras mayúsculas vs. minúsculas) asegurando que pertenezcan a la misma línea horizontal.



In [28]:
# ==============================================================================
# 5.1 MÓDULO DE EXTRACCIÓN ESPACIAL
# ==============================================================================

def extract_spatial_tokens(hocr_file_path: str) -> pd.DataFrame:
    """
    Analiza un archivo HOCR, extrae tokens de texto y calcula sus centroides geométricos.

    Argumentos:
        hocr_file_path (str): Ruta absoluta al archivo .hocr de origen.

    Retorna:
        pd.DataFrame: Estructura tabular con 'text', 'x_start' y 'y_centroid' para cada token.
    """
    with open(hocr_file_path, 'r', encoding='utf-8') as html_file:
        parsed_soup = BeautifulSoup(html_file, 'html.parser', from_encoding='utf-8')

    extracted_tokens = []

    # Búsqueda de todos los nodos clasificados como palabras
    for word_node in parsed_soup.find_all('span', class_='ocrx_word'):
        raw_text = word_node.get_text(strip=True)
        if not raw_text:
            continue

        # Captura de coordenadas de la caja delimitadora (bbox)
        bbox_pattern = re.search(r'bbox (\d+) (\d+) (\d+) (\d+)', word_node.get('title', ''))
        if bbox_pattern:
            x_min, y_min, x_max, y_max = map(int, bbox_pattern.groups())
            extracted_tokens.append({
                'text': raw_text,
                'x_start': x_min,
                'y_centroid': (y_min + y_max) / 2 # Normalización de altura
            })

    return pd.DataFrame(extracted_tokens)

### 5.2. Módulo de Reconstrucción Topológica (`cluster_grid_topology`)
Una vez extraídos los nodos en un espacio continuo 2D, es necesario discretizarlos en una topología de cuadrícula (filas y columnas).

**Justificación Algorítmica:**

* **Agrupamiento Bidimensional Independiente:** Se aplica el algoritmo DBSCAN dos veces. Primero sobre el eje Y para detectar la densidad de las filas, y luego sobre el eje X para detectar las columnas.
* **Tolerancia Dinámica (Epsilon):** Los hiperparámetros `y_tolerance` (15px) y `x_tolerance` (50px) representan la distancia máxima permisible entre palabras. Esto permite agrupar palabras que visualmente pertenecen a la misma celda, incluso si el documento original sufrió una distorsión o inclinación durante el escaneo.
* **Mapeo de Intersección:** La matriz final se construye encontrando la intersección lógica entre los identificadores de clúster de fila y columna, ensamblando las celdas de izquierda a derecha y de arriba hacia abajo.

In [29]:
# ==============================================================================
# 5.2 MÓDULO DE RECONSTRUCCIÓN TOPOLÓGICA
# ==============================================================================

def cluster_grid_topology(spatial_dataframe: pd.DataFrame, y_tolerance: int = 15, x_tolerance: int = 50) -> str:
    """
    Reconstruye la cuadrícula tabular 2D utilizando el agrupamiento espacial DBSCAN.

    Argumentos:
        spatial_dataframe (pd.DataFrame): Dataframe con las coordenadas extraídas.
        y_tolerance (int): Tolerancia vertical (píxeles) para agrupar en filas.
        x_tolerance (int): Tolerancia horizontal (píxeles) para agrupar en columnas.

    Retorna:
        str: Representación CSV cruda basada puramente en geometría espacial.
    """
    if spatial_dataframe.empty:
        return ""

    # 1. Agrupamiento de Filas (Eje Y)
    clusterer_y = DBSCAN(eps=y_tolerance, min_samples=1).fit(spatial_dataframe[['y_centroid']])
    spatial_dataframe['row_cluster_id'] = clusterer_y.labels_

    # Ordenar filas geométricamente de arriba a abajo
    spatial_dataframe['row_logical_order'] = spatial_dataframe['row_cluster_id'].map(
        spatial_dataframe.groupby('row_cluster_id')['y_centroid'].mean()
    )
    spatial_dataframe = spatial_dataframe.sort_values(by=['row_logical_order', 'x_start'])

    # 2. Agrupamiento de Columnas (Eje X)
    clusterer_x = DBSCAN(eps=x_tolerance, min_samples=1).fit(spatial_dataframe[['x_start']])
    spatial_dataframe['col_cluster_id'] = clusterer_x.labels_

    # Ordenar columnas de izquierda a derecha
    spatial_dataframe['col_logical_order'] = spatial_dataframe['col_cluster_id'].map(
        spatial_dataframe.groupby('col_cluster_id')['x_start'].mean()
    )

    # 3. Intersección Matricial
    reconstructed_matrix = []
    sorted_unique_rows = sorted(spatial_dataframe['row_logical_order'].unique())
    sorted_unique_cols = sorted(spatial_dataframe['col_logical_order'].unique())

    for row_idx in sorted_unique_rows:
        current_row_data = spatial_dataframe[spatial_dataframe['row_logical_order'] == row_idx]

        # Ensamblar celdas uniendo palabras que comparten el mismo clúster de fila y columna
        row_cells = [
            " ".join(current_row_data[current_row_data['col_logical_order'] == col_idx]['text'].tolist())
            for col_idx in sorted_unique_cols
        ]
        reconstructed_matrix.append(row_cells)

    return "\n".join([",".join(row) for row in reconstructed_matrix])

### 5.3. Módulo de Refinamiento Semántico (`refine_semantics_with_llm`)
La topología espacial es precisa, pero la lectura óptica a nivel de carácter es propensa a fallos (ej. ruido de sal y pimienta en el escaneo).

**Justificación de IA Generativa:**
* **Inferencia de Contexto:** A diferencia de un corrector ortográfico tradicional, el LLM puede inferir que si una columna completa contiene fechas como "2025", "2024", la celda leída como "2O23" por el OCR debe corregirse semánticamente a "2023".
* **Manejo de Errores (Resiliencia):** Se implementó un bloque `try-except` crítico. Si la llamada a la API agota su tiempo de espera o falla la autenticación, el sistema no colapsa; degrada suavemente devolviendo la tabla geométrica cruda generada por DBSCAN, garantizando siempre una salida para el usuario.

In [30]:
# ==============================================================================
# 5.3 MÓDULO DE REFINAMIENTO SEMÁNTICO (LLM)
# ==============================================================================

def refine_semantics_with_llm(raw_csv_string: str, target_model: str) -> str:
    """
    Transductor semántico que corrige errores tipográficos de OCR vía LLM.

    Argumentos:
        raw_csv_string (str): Matriz espacial cruda generada por DBSCAN.
        target_model (str): Identificador del modelo Gemini a instanciar.

    Retorna:
        str: Cadena CSV estructurada y corregida semánticamente.
    """
    llm_instance = genai.GenerativeModel(target_model)

    # Ingeniería de Prompt Robusta (Zero-shot con Reglas Estrictas de Sanitización CSV)
    system_prompt = (
        "Actúa como un parser de datos determinista. Tu ÚNICA función es recibir una matriz espacial cruda "
        "y devolver el texto formateado en CSV puro. "
        "CUMPLE OBLIGATORIAMENTE ESTAS REGLAS ESTRICTAS:\n\n"
        "1. PRESERVACIÓN ABSOLUTA: NO elimines texto. Títulos, nombres de empresas y notas DEBEN conservarse.\n"
        "2. ESCAPE DE TEXTO (TEXT WRAPPING): Esta es una regla CRÍTICA. Si un texto, título, dirección o nombre de empresa "
        "contiene comas gramaticales (ej. 'Almidones Mexicanos, S. A. de C. V.' o 'Sinaloa, S.A.'), "
        "TIENES que encerrar ese bloque de texto completo entre comillas dobles en tu salida CSV "
        "(ej. \"Almidones Mexicanos, S. A. de C. V.\"). Esto evitará que la coma rompa la columna.\n"
        "3. MANEJO DE ASIMETRÍA: Títulos o categorías que agrupan filas inferiores van en la primera columna, "
        "seguidos de comas vacías (ej. '\"TÍTULO PRINCIPAL\",,,,\n').\n"
        "4. NÚMEROS LIMPIOS: Elimina TODAS las comas que actúen como separadores de miles en las cifras numéricas "
        "(escribe '101606' en lugar de '101,606'). FUSIONA números fragmentados.\n"
        "5. CONSISTENCIA MONETARIA: Añade el símbolo '$' al inicio de TODOS los valores financieros "
        "(ej. '$101606'). NO dejes el símbolo en una columna separada.\n"
        "6. ALINEACIÓN VERTICAL ESTRICTA: Usa comas vacías (ej. ',,$150000') para empujar los datos numéricos "
        "hacia la derecha si falta la categoría izquierda, asegurando que caigan bajo su columna correcta (ej. su Año).\n"
        "7. CANDADO DE FORMATO: PROHIBIDO escribir código de programación. PROHIBIDO repetir estas instrucciones. "
        "PROHIBIDO explicar lo que hiciste. Devuelve ÚNICAMENTE los datos procesados en texto plano CSV.\n\n"
        f"Datos de entrada espacial cruda:\n{raw_csv_string}"
    )

    try:
        api_response = llm_instance.generate_content(system_prompt)
        # Limpieza robusta de la salida generada
        cleaned_csv = api_response.text.replace('```csv', '').replace('```', '').strip()
        return cleaned_csv
    except Exception as llm_error:
        # Mecanismo de degradación elegante (Graceful degradation)
        print(f"Advertencia: Fallo en la inferencia LLM ({llm_error}). Retornando datos espaciales.")
        return raw_csv_string

## 6. Orquestación por Lotes (Batch Processing) y Visualización
Para consolidar la automatización, este controlador maestro escanea el directorio montado en busca de todos los archivos `.hocr`. Posteriormente, ejecuta de manera secuencial los módulos de extracción, topología y semántica sobre cada archivo, iterando todo el flujo de trabajo.

**Manejo Avanzado de Codificación (BOM para Excel):**
Durante la persistencia de los archivos en la nube, se ha implementado de manera deliberada la codificación `utf-8-sig` para la escritura del CSV. Esta decisión inyecta un *Byte Order Mark* (BOM) invisible al inicio del archivo. Con esta técnica de ingeniería a bajo nivel, se previene el fenómeno de corrupción de caracteres (*Mojibake*), forzando a que programas de hoja de cálculo que asumen codificaciones locales (como Microsoft Excel) rendericen perfectamente los caracteres especiales del idioma español (acentos y la letra 'ñ') extraídos del documento original sin requerir procesamiento adicional del LLM.

Al finalizar, el orquestador genera *dataframes* de previsualización interactiva con Pandas para auditoría rápida y dispara las descargas directas de los resultados refinados a la máquina local del operador.

In [31]:
# ==============================================================================
# 6. ORQUESTADOR MAESTRO DE PROCESAMIENTO POR LOTES
# ==============================================================================

# Exploración del directorio objetivo
hocr_file_list = glob.glob(os.path.join(PROJECT_FOLDER, '*.hocr'))

if not hocr_file_list:
    print(f"⚠️ Advertencia: No se encontraron archivos .hocr en: {PROJECT_FOLDER}")
else:
    total_files = len(hocr_file_list)
    print(f"Se detectaron {total_files} archivos para procesar en lote.\n")
    print("-" * 60)

    for file_path in hocr_file_list:
        base_name = os.path.basename(file_path).replace('.hocr', '')
        output_csv_path = os.path.join(OUTPUT_DIRECTORY, f"{base_name}_reconstructed.csv")

        print(f"🔄 Procesando documento: {base_name}.hocr ...")

        # Ejecución del pipeline modular
        spatial_dataframe = extract_spatial_tokens(file_path)
        raw_csv_string = cluster_grid_topology(spatial_dataframe)
        refined_csv_string = refine_semantics_with_llm(raw_csv_string, TARGET_LLM_MODEL)

        # Persistencia en la nube (Google Drive) con BOM para Excel
        with open(output_csv_path, 'w', encoding='utf-8-sig') as output_file:
            output_file.write(refined_csv_string)

        print(f"✅ Reconstrucción exitosa.")

        # Auditoría visual
        try:
            preview_dataframe = pd.read_csv(output_csv_path, sep=',', header=None)
            print(f"\n📊 Auditoría visual de {base_name}_reconstructed.csv:")
            display(preview_dataframe.head())
        except Exception as preview_error:
            print(f"No fue posible renderizar la auditoría visual: {preview_error}")

        # Extracción local
        print("⬇️ Solicitando descarga del artefacto...")
        files.download(output_csv_path)

        print("-" * 60)

    print("\n🎉 Pipeline de procesamiento finalizado correctamente.")

Se detectaron 3 archivos para procesar en lote.

------------------------------------------------------------
🔄 Procesando documento: 3T_4.hocr ...
✅ Reconstrucción exitosa.

📊 Auditoría visual de 3T_4_reconstructed.csv:


,0,1,2,3,4,5
0,"GRANJAS MARINAS DE SINALOA, S.A. DE C.V.",NaN,NaN,NaN,NaN,NaN
1,ESTADO DE POSICIÓN FINANCIERA,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,AL 31 DE DICIEMBRE DE :,NaN,NaN,NaN
3,NaN,NaN,"( Cifras expresadas,en pesos )",NaN,NaN,NaN
4,ACTIVO,NaN,NaN,NaN,2020,2019


⬇️ Solicitando descarga del artefacto...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,BERL
1,NaN,NaN,NaN,"GRUPO,AGROINDUSTRIAL DE TAMAULIPAS , S DE PR D...",NaN,NaN
2,NaN,NaN,NaN,NaN,"CARR . RIBEREÑA KM 34 S / N , ZONA INDUSTRIAL ...",NaN
3,NaN,NaN,NaN,NaN,"CD . GUSTAVO DIAZ ORDAZ , TAM . , TELS .: 891 ...",NaN
4,NaN,NaN,NaN,NaN,BALANCE GENERAL AL 31 DE DICIEMBRE DE 2021 .,NaN


⬇️ Solicitando descarga del artefacto...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

------------------------------------------------------------

🎉 Pipeline de procesamiento finalizado correctamente.
